![walmartecomm](walmartecomm.jpg)
# 🛒 Building a Retail Data Pipeline
**Auteur :** Yoann LEHONG CHEFFSON 
[![LinkedIn](https://img.shields.io/badge/LinkedIn-0077B5?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/yoann-lehong-cheffson/) [![GitHub](https://img.shields.io/badge/GitHub-100000?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Yoannlcf/My-Data-Journey)

---

## 📋 Contexte du Projet
Réalisé dans le cadre de ma spécialisation intensive en Data Engineering et Cloud Computing, ce projet illustre la conception et l'implémentation d'un **pipeline ETL (Extract, Transform, Load)** robuste. L'objectif est de consolider et de préparer des données de ventes hétérogènes pour la multinationale du retail, Walmart. 

Dans un écosystème Data moderne, la maîtrise des flux de données est primordiale pour garantir la fiabilité des analyses. Ce notebook démontre de manière modulaire les étapes suivantes :
1. **Extract :** Récupération de données depuis des sources multiples (fichiers CSV simulant un export SQL et fichiers optimisés Parquet).
2. **Transform :** Nettoyage et préparation des données via `pandas` (gestion des valeurs manquantes, feature engineering sur les dates, filtrage de performance et optimisation du schéma).
3. **Load :** Sauvegarde des données propres et agrégées dans un format standardisé, prêtes à être ingérées par une équipe de Data Analysts ou un outil de Business Intelligence.

**🛠️ Stack Technique :** Python, Pandas, manipulation de formats (CSV, Parquet).

## 1. Étape d'Extraction (Extract)
Dans cette première étape, nous créons la fonction `extract()` qui a pour rôle de :
* Lire la table principale des ventes (`grocery_sales.csv`, simulant un export SQL).
* Lire les données complémentaires au format orienté colonne (`extra_data.parquet`).
* Fusionner ces deux sources de données brutes sur leur index commun.

In [1]:
import pandas as pd

def extract(grocery_sales_path, extra_data_path):
    # 1. Charger le CSV 
    grocery_sales = pd.read_csv(grocery_sales_path)
    
    # 2. Charger le fichier Parquet
    extra_data = pd.read_parquet(extra_data_path)

    # 3. Fusionner les deux DataFrames
    merged_df = grocery_sales.merge(extra_data, on="index")

    return merged_df

merged_df = extract("data/grocery_sales.csv", "data/extra_data.parquet")

merged_df.head()

,index,Store_ID,Date,Dept,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
0,0,1,2010-02-05,1,24924.50,0,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,3.0,151315.0
1,1,1,2010-02-05,26,11737.12,0,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,3.0,151315.0
2,2,1,2010-02-05,17,13223.76,0,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,3.0,151315.0
3,3,1,2010-02-05,45,37.44,0,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,NaN,3.0,151315.0
4,4,1,2010-02-05,28,1085.29,0,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,NaN,3.0,151315.0


## 2. Étape de Transformation (Transform)
Les données brutes nécessitent un nettoyage rigoureux avant de pouvoir en extraire des insights. La fonction `transform()` va appliquer les règles de gestion suivantes :
1. **Gestion des valeurs manquantes (Imputation)** : Remplacement des valeurs nulles par la moyenne de leur colonne respective (`CPI`, `Weekly_Sales`, `Unemployment`) afin de maintenir la cohérence statistique et de ne pas fausser l'analyse.
2. **Feature Engineering & Optimisation** : Typage explicite de la colonne de date (au format `%Y-%m-%d`) pour accélérer le temps de traitement, puis extraction du mois pour faciliter les agrégations futures.
3. **Filtrage** : Conservation exclusive des lignes où les ventes hebdomadaires (`Weekly_Sales`) dépassent 10 000 $.
4. **Nettoyage du schéma** : Suppression ciblée des colonnes devenues inutiles pour alléger le jeu de données final et économiser de la mémoire.

In [2]:
def transform(raw_data):
    # 1. Imputation statistique 
    raw_data.fillna(
        {
            'CPI': raw_data['CPI'].mean(),
            'Weekly_Sales': raw_data['Weekly_Sales'].mean(),
            'Unemployment': raw_data['Unemployment'].mean(),
        }, inplace=True
    )

    # 2. Conversion de la date avec format explicite (Optimisation des performances)
    raw_data["Date"] = pd.to_datetime(raw_data["Date"], format="%Y-%m-%d")
    raw_data["Month"] = raw_data["Date"].dt.month

    # 3. Filtrage avec .loc (Explicite et propre)
    clean_data = raw_data.loc[raw_data["Weekly_Sales"] > 10000, :]

    # 4. Suppression des colonnes inutiles
    columns_to_drop = ["index", "Temperature", "Fuel_Price", "MarkDown1", "MarkDown2", 
                       "MarkDown3", "MarkDown4", "MarkDown5", "Type", "Size", "Date"]
    clean_data = clean_data.drop(columns=columns_to_drop, errors='ignore')
    
    return clean_data

# Appel de la fonction
clean_data = transform(merged_df)
clean_data.head()

,Store_ID,Dept,Weekly_Sales,IsHoliday,CPI,Unemployment,Month
0,1,1,24924.50,0,211.096358,8.106000,2.0
1,1,26,11737.12,0,211.096358,8.106000,2.0
2,1,17,13223.76,0,211.096358,8.106000,2.0
5,1,79,46729.77,0,211.096358,7.500052,2.0
6,1,55,21249.31,0,211.096358,7.500052,2.0


## 3. Agrégation des données (Analyze / Transform)
Une fois les données nettoyées, nous devons préparer une vue agrégée pour les tableaux de bord métiers. 
La fonction `avg_weekly_sales_per_month()` utilise le **chaînage de méthodes** (method chaining) pour calculer la moyenne des ventes hebdomadaires par mois.

In [3]:
def avg_weekly_sales_per_month(clean_data):
    # On sélectionne les colonnes, on groupe, on agrège, on réinitialise l'index et on arrondit
    agg_data = (
        clean_data[['Month','Weekly_Sales']]
        .groupby('Month')
        .agg(Avg_Sales = ('Weekly_Sales','mean'))
        .reset_index()
        .round(2)
    )

    return agg_data

agg_data = avg_weekly_sales_per_month(clean_data)

agg_data

,Month,Avg_Sales
0,1.0,33174.18
1,2.0,34333.33
2,3.0,33220.89
3,4.0,33392.37
4,5.0,33339.89
5,6.0,34582.47
6,7.0,33922.76
7,8.0,33644.79
8,9.0,33258.05
9,10.0,32736.99


## 4. Étape de Chargement (Load)
Dernière étape du pipeline ETL : la persistance des données. 
La fonction `load()` exporte le jeu de données nettoyé ainsi que la vue agrégée dans le dossier de destination au format CSV, prêts à être ingérés par un outil de Business Intelligence ou une base de données analytique.

In [4]:
def load(clean_data, agg_data, clean_path, agg_path):
    clean_data.to_csv(clean_path, index = False)
    agg_data.to_csv(agg_path, index = False)

# On crée les fichiers dans le dossier data
load(clean_data, agg_data, "data/clean_data.csv", "data/agg_data.csv")
print("Fichiers sauvegardés avec succès !")

Fichiers sauvegardés avec succès !


## 5. Validation du Pipeline
Un bon pipeline de données doit intégrer des tests de validation pour s'assurer que les tâches se sont déroulées sans erreur. La fonction `validation()` vérifie l'existence physique des fichiers générés.

In [5]:
import os

def validation(file_path):
    # Vérifie si le fichier existe
    if not os.path.exists(file_path):
        raise Exception(f"Erreur critique : Le fichier est introuvable au chemin '{file_path}'")
    else:
        print(f"Succès : Fichier validé -> {file_path}")

validation("data/clean_data.csv")
validation("data/agg_data.csv")

Succès : Fichier validé -> data/clean_data.csv
Succès : Fichier validé -> data/agg_data.csv
